In [1]:
!pip install spotpy

In [ ]:
### NSE

In [1]:
import os
import datetime
from pathlib import Path
import xarray as xr
import pandas as pd
from numpy import mean, std, corrcoef
from math import sqrt
import spotpy
import psutil
from hydrogr import InputDataHandler, ModelGr6j
from joblib import Parallel, delayed

# --- Function to run one replicate calibration for a catchment ---
def calibrate_catchment_replicate(nc_file, n_samples, cal_start, cal_end, val_start, val_end, replicate_id, output_dir):
    catchment_name = Path(nc_file).stem
    catchment_outdir = Path(output_dir) / catchment_name
    catchment_outdir.mkdir(parents=True, exist_ok=True)

    # Load NetCDF
    ds = xr.open_dataset(nc_file)
    df = ds.to_dataframe().reset_index()

    df = df.rename(columns={
        "total_precipitation_sum": "precipitation",
        "temperature": "temperature",
        "potential_evaporation_sum": "evapotranspiration",
        "discharge_spec": "flow_mm"
    })
    df["date"] = pd.to_datetime(df["date"])
    df.index = df["date"]

    # Calibration period
    cal_mask = (df['date'] >= cal_start) & (df['date'] <= cal_end)
    calibration_data = df.loc[cal_mask]

    # Spotpy setup
    class SpotpySetup(object):
        def __init__(self, data):
            self.data = data
            self.model_inputs = InputDataHandler(ModelGr6j, self.data)
            self.params = [
                spotpy.parameter.Uniform("x1", 100, 1400),
                spotpy.parameter.Uniform("x2", -4, 4),
                spotpy.parameter.Uniform("x3", 0, 500),
                spotpy.parameter.Uniform("x4", 0, 10),
                spotpy.parameter.Uniform("x5", -4, 4),
                spotpy.parameter.Uniform("x6", 0, 20),
            ]

        def parameters(self):
            return spotpy.parameter.generate(self.params)

        def simulation(self, vector):
            return self._run(*vector)

        def evaluation(self):
            return self.data["flow_mm"].values

        def objectivefunction(self, simulation, evaluation):
            return -sqrt(mean((simulation - evaluation)**2))

        def _run(self, x1, x2, x3, x4, x5, x6):
            parameters = {"X1": x1, "X2": x2, "X3": x3, "X4": x4, "X5": x5, "X6": x6}
            model = ModelGr6j(parameters)
            outputs = model.run(self.model_inputs.data)
            return outputs["flow"].values

    # Run Spotpy calibration
    spotpy_setup = SpotpySetup(calibration_data)
    sampler = spotpy.algorithms.sceua(spotpy_setup, dbformat="ram")
    sampler.sample(n_samples)
    results = sampler.getdata()
    best_parameters = spotpy.analyser.get_best_parameterset(results)
    params_list = list(best_parameters[0])
    best_params = dict(zip(["X1","X2","X3","X4","X5","X6"], params_list))

    # --- Calibration performance ---
    model = ModelGr6j(best_params)
    cal_outputs = model.run(calibration_data)
    cal_df = calibration_data.copy()
    cal_df["simulated_flow"] = cal_outputs["flow"].values
    cal_df.to_csv(catchment_outdir / f"{catchment_name}_calibration_replicate{replicate_id}.csv", index=False)

    obs_cal = cal_df["flow_mm"].values
    sim_cal = cal_df["simulated_flow"].values
    nse_cal = 1 - sum((sim_cal - obs_cal)**2) / sum((obs_cal - mean(obs_cal))**2)
    r = corrcoef(sim_cal, obs_cal)[0, 1]
    alpha = std(sim_cal) / std(obs_cal)
    beta = mean(sim_cal) / mean(obs_cal)
    kge_cal = 1 - sqrt((r-1)**2 + (alpha-1)**2 + (beta-1)**2)

    # --- Validation performance ---
    val_mask = (df['date'] >= val_start) & (df['date'] <= val_end)
    validation_data = df.loc[val_mask]
    val_outputs = model.run(validation_data)
    val_df = validation_data.copy()
    val_df["simulated_flow"] = val_outputs["flow"].values
    val_df.to_csv(catchment_outdir / f"{catchment_name}_validation_replicate{replicate_id}.csv", index=False)

    obs_val = val_df["flow_mm"].values
    sim_val = val_df["simulated_flow"].values
    nse_val = 1 - sum((sim_val - obs_val)**2) / sum((obs_val - mean(obs_val))**2)
    r = corrcoef(sim_val, obs_val)[0, 1]
    alpha = std(sim_val) / std(obs_val)
    beta = mean(sim_val) / mean(obs_val)
    kge_val = 1 - sqrt((r-1)**2 + (alpha-1)**2 + (beta-1)**2)

    return {
        "catchment": catchment_name,
        "replicate": replicate_id,
        **best_params,
        "NSE_cal": nse_cal,
        "KGE_cal": kge_cal,
        "NSE_val": nse_val,
        "KGE_val": kge_val
    }


# -------------------------
# Auto-tune Parallel Execution
# -------------------------
def get_safe_njobs(min_ram_per_job_gb=2.5):
    physical_cores = psutil.cpu_count(logical=False)
    logical_cores = psutil.cpu_count(logical=True)
    total_ram_gb = psutil.virtual_memory().total / (1024 ** 3)
    available_ram_gb = psutil.virtual_memory().available / (1024 ** 3)
    max_jobs_by_memory = int(available_ram_gb // min_ram_per_job_gb)
    safe_jobs = min(physical_cores, max_jobs_by_memory)
    safe_jobs = max(1, safe_jobs)

    print(f"🧠 {physical_cores} physical cores / {logical_cores} logical cores detected.")
    print(f"💾 Total RAM: {total_ram_gb:.1f} GB | Available: {available_ram_gb:.1f} GB")
    print(f"⚙️  Using {safe_jobs} parallel jobs (~{min_ram_per_job_gb} GB/job)")
    return safe_jobs




# --- Main: Parallel execution with replicates ---
data_path = Path.cwd().parent / "data" / "GB" ##
output_dir = Path.cwd() / "GR6J_CAL"  # <--- Define output directory
output_dir.mkdir(parents=True, exist_ok=True)
nc_files = list(data_path.glob("*.nc"))

n_samples = 12000
n_replicates = 10 #10
#n_jobs = -1  # Use all cores

n_jobs = get_safe_njobs(min_ram_per_job_gb=2.5)###min_ram_per_job_gb=2.5
all_results = Parallel(n_jobs=n_jobs, verbose=5)(
    delayed(calibrate_catchment_replicate)(
        nc_file, n_samples,
        datetime.datetime(1999,1,1),
        datetime.datetime(2008,12,31),
        datetime.datetime(2009,1,1),
        datetime.datetime(2014,12,31),
        replicate_id,
        output_dir   # ✅ Added missing argument here
    )
    for nc_file in nc_files
    for replicate_id in range(1, n_replicates+1)
)

# --- Save overall summary ---
summary_df = pd.DataFrame(all_results)

# Save per-catchment replicate results
for catchment, group in summary_df.groupby("catchment"):
    catchment_outdir = output_dir / catchment
    catchment_outdir.mkdir(parents=True, exist_ok=True)
    catchment_summary_path = catchment_outdir / f"{catchment}_replicates_summary.csv"
    group.to_csv(catchment_summary_path, index=False)
    print(f"✅ Saved {catchment} replicate summary to: {catchment_summary_path}")

# Also save combined summary for all catchments (optional)
summary_path = output_dir / "best_parameters_GR6J_replicates.csv"
summary_df.to_csv(summary_path, index=False)
print(f"\n📄 Global summary saved to: {summary_path}")



🧠 16 physical cores / 24 logical cores detected.
💾 Total RAM: 63.7 GB | Available: 46.6 GB
⚙️  Using 16 parallel jobs (~2.5 GB/job)


[Parallel(n_jobs=16)]: Using backend LokyBackend with 16 concurrent workers.
[Parallel(n_jobs=16)]: Done  40 tasks      | elapsed:   57.8s
[Parallel(n_jobs=16)]: Done 130 tasks      | elapsed:  2.8min
[Parallel(n_jobs=16)]: Done 256 tasks      | elapsed:  5.0min
[Parallel(n_jobs=16)]: Done 418 tasks      | elapsed:  8.2min
[Parallel(n_jobs=16)]: Done 616 tasks      | elapsed: 11.8min
[Parallel(n_jobs=16)]: Done 850 tasks      | elapsed: 16.4min
[Parallel(n_jobs=16)]: Done 1120 tasks      | elapsed: 21.5min
[Parallel(n_jobs=16)]: Done 1426 tasks      | elapsed: 27.6min
[Parallel(n_jobs=16)]: Done 1768 tasks      | elapsed: 34.3min
[Parallel(n_jobs=16)]: Done 2146 tasks      | elapsed: 41.6min
[Parallel(n_jobs=16)]: Done 2560 tasks      | elapsed: 49.7min
[Parallel(n_jobs=16)]: Done 3010 tasks      | elapsed: 58.4min
[Parallel(n_jobs=16)]: Done 3496 tasks      | elapsed: 67.9min
[Parallel(n_jobs=16)]: Done 4018 tasks      | elapsed: 78.2min
[Parallel(n_jobs=16)]: Done 4576 tasks      | e

✅ Saved 10002 replicate summary to: C:\Users\jougahi001\OneDrive - University of Dundee\Hydrogr\hydrogr-master\hydrogr-master\example\GR6J_CAL\10002\10002_replicates_summary.csv
✅ Saved 10003 replicate summary to: C:\Users\jougahi001\OneDrive - University of Dundee\Hydrogr\hydrogr-master\hydrogr-master\example\GR6J_CAL\10003\10003_replicates_summary.csv
✅ Saved 1001 replicate summary to: C:\Users\jougahi001\OneDrive - University of Dundee\Hydrogr\hydrogr-master\hydrogr-master\example\GR6J_CAL\1001\1001_replicates_summary.csv
✅ Saved 101002 replicate summary to: C:\Users\jougahi001\OneDrive - University of Dundee\Hydrogr\hydrogr-master\hydrogr-master\example\GR6J_CAL\101002\101002_replicates_summary.csv
✅ Saved 101005 replicate summary to: C:\Users\jougahi001\OneDrive - University of Dundee\Hydrogr\hydrogr-master\hydrogr-master\example\GR6J_CAL\101005\101005_replicates_summary.csv
✅ Saved 102001 replicate summary to: C:\Users\jougahi001\OneDrive - University of Dundee\Hydrogr\hydrogr-ma

In [5]:
#!pip install psutil

In [ ]:
#### GR4J NSE and logNSE

In [1]:
import os
import datetime
from pathlib import Path
import xarray as xr
import pandas as pd
import numpy as np
from numpy import mean, std, corrcoef
from math import sqrt
import spotpy
import psutil
from hydrogr import InputDataHandler, ModelGr4j
from joblib import Parallel, delayed

# -----------------------------
# Spotpy setup for GR6J
# -----------------------------
class SpotpySetup(object):
    def __init__(self, data):
        self.data = data
        self.model_inputs = InputDataHandler(ModelGr4j, self.data)
        # Define GR6J parameters
        self.params = [
            spotpy.parameter.Uniform("x1", 100, 1400),
            spotpy.parameter.Uniform("x2", -4, 4),
            spotpy.parameter.Uniform("x3", 0, 500),
            spotpy.parameter.Uniform("x4", 0, 10),
        ]

    def parameters(self):
        return spotpy.parameter.generate(self.params)

    def simulation(self, vector):
        return self._run(*vector)

    def evaluation(self):
        return self.data["flow_mm"].values

    def objectivefunction(self, simulation, evaluation):
        """
        Multi-objective: maximize NSE and logNSE simultaneously.
        """
        mask = (evaluation > 0) & (simulation > 0)
        if mask.sum() == 0:
            return [-9999, -9999]

        obs_valid = evaluation[mask]
        sim_valid = simulation[mask]

        # Standard NSE
        nse = 1 - np.sum((sim_valid - obs_valid) ** 2) / np.sum((obs_valid - np.mean(obs_valid)) ** 2)
        # LogNSE
        log_nse = 1 - np.sum((np.log(sim_valid) - np.log(obs_valid)) ** 2) / np.sum((np.log(obs_valid) - np.mean(np.log(obs_valid))) ** 2)

        return [nse, log_nse]

    def _run(self, x1, x2, x3, x4):
        parameters = {"X1": x1, "X2": x2, "X3": x3, "X4": x4}
        model = ModelGr4j(parameters)
        outputs = model.run(self.model_inputs.data)
        return outputs["flow"].values

# -----------------------------
# Function to calibrate one catchment replicate
# -----------------------------
def calibrate_catchment_replicate(nc_file, n_samples, cal_start, cal_end, val_start, val_end, replicate_id, output_dir):
    catchment_name = Path(nc_file).stem
    catchment_outdir = Path(output_dir) / catchment_name
    catchment_outdir.mkdir(parents=True, exist_ok=True)

    # Load NetCDF
    ds = xr.open_dataset(nc_file)
    df = ds.to_dataframe().reset_index()
    df = df.rename(columns={
        "total_precipitation_sum": "precipitation",
        "temperature": "temperature",
        "potential_evaporation_sum": "evapotranspiration",
        "discharge_spec": "flow_mm"
    })
    df["date"] = pd.to_datetime(df["date"])
    df.index = df["date"]

    # Calibration period
    cal_mask = (df['date'] >= cal_start) & (df['date'] <= cal_end)
    calibration_data = df.loc[cal_mask]

    # Run Spotpy calibration
    spotpy_setup = SpotpySetup(calibration_data)
    sampler = spotpy.algorithms.sceua(spotpy_setup, dbformat="ram")
    sampler.sample(n_samples)
    results = sampler.getdata()

    # --- Robust handling of structured array / DataFrame ---
    results_df = pd.DataFrame(results)  # Converts structured array to DataFrame if needed

    # List all parameter columns (par0-par5 or x1-x6)
    param_cols = [col for col in results_df.columns if col.startswith("par") or col.lower() in ["x1","x2","x3","x4"]]

    # Select best replicate based on logNSE (usually like1)
    best_idx = np.argmax(results_df['like1'])
    best_params = {f"X{i+1}": results_df[param_cols[i]][best_idx] for i in range(len(param_cols))}

    # --- Calibration simulation and metrics ---
    model = ModelGr4j(best_params)
    cal_outputs = model.run(calibration_data)
    cal_df = calibration_data.copy()
    cal_df["simulated_flow"] = cal_outputs["flow"].values
    cal_df.to_csv(catchment_outdir / f"{catchment_name}_calibration_replicate{replicate_id}.csv", index=False)

    obs_cal = cal_df["flow_mm"].values
    sim_cal = cal_df["simulated_flow"].values
    mask_cal = (obs_cal > 0) & (sim_cal > 0)
    nse_cal = 1 - np.sum((sim_cal - obs_cal)**2) / np.sum((obs_cal - np.mean(obs_cal))**2)
    log_nse_cal = 1 - np.sum((np.log(sim_cal[mask_cal]) - np.log(obs_cal[mask_cal]))**2) / np.sum((np.log(obs_cal[mask_cal]) - np.mean(np.log(obs_cal[mask_cal])))**2)
    r = corrcoef(sim_cal, obs_cal)[0, 1]
    alpha = std(sim_cal) / std(obs_cal)
    beta = mean(sim_cal) / mean(obs_cal)
    kge_cal = 1 - sqrt((r-1)**2 + (alpha-1)**2 + (beta-1)**2)

    # --- Validation simulation and metrics ---
    val_mask = (df['date'] >= val_start) & (df['date'] <= val_end)
    validation_data = df.loc[val_mask]
    val_outputs = model.run(validation_data)
    val_df = validation_data.copy()
    val_df["simulated_flow"] = val_outputs["flow"].values
    val_df.to_csv(catchment_outdir / f"{catchment_name}_validation_replicate{replicate_id}.csv", index=False)

    obs_val = val_df["flow_mm"].values
    sim_val = val_df["simulated_flow"].values
    mask_val = (obs_val > 0) & (sim_val > 0)
    nse_val = 1 - np.sum((sim_val - obs_val)**2) / np.sum((obs_val - np.mean(obs_val))**2)
    log_nse_val = 1 - np.sum((np.log(sim_val[mask_val]) - np.log(obs_val[mask_val]))**2) / np.sum((np.log(obs_val[mask_val]) - np.mean(np.log(obs_val[mask_val])))**2)
    r = corrcoef(sim_val, obs_val)[0, 1]
    alpha = std(sim_val) / std(obs_val)
    beta = mean(sim_val) / mean(obs_val)
    kge_val = 1 - sqrt((r-1)**2 + (alpha-1)**2 + (beta-1)**2)

    # --- Return metrics and best parameters ---
    return {
        "catchment": catchment_name,
        "replicate": replicate_id,
        **best_params,
        "NSE_cal": nse_cal,
        "logNSE_cal": log_nse_cal,
        "KGE_cal": kge_cal,
        "NSE_val": nse_val,
        "logNSE_val": log_nse_val,
        "KGE_val": kge_val
    }

# -----------------------------
# Function to detect safe number of parallel jobs
# -----------------------------
def get_safe_njobs(min_ram_per_job_gb=2.5):
    physical_cores = psutil.cpu_count(logical=False)
    logical_cores = psutil.cpu_count(logical=True)
    total_ram_gb = psutil.virtual_memory().total / (1024 ** 3)
    available_ram_gb = psutil.virtual_memory().available / (1024 ** 3)
    max_jobs_by_memory = int(available_ram_gb // min_ram_per_job_gb)
    safe_jobs = min(physical_cores, max_jobs_by_memory)
    safe_jobs = max(1, safe_jobs)
    print(f"🧠 {physical_cores} physical cores / {logical_cores} logical cores detected.")
    print(f"💾 Total RAM: {total_ram_gb:.1f} GB | Available: {available_ram_gb:.1f} GB")
    print(f"⚙️  Using {safe_jobs} parallel jobs (~{min_ram_per_job_gb} GB/job)")
    return safe_jobs

# -----------------------------
# Main execution
# -----------------------------
data_path = Path.cwd().parent / "data" / "GB"  # Adjust your data folder
output_dir = Path.cwd() / "GR4J_Calib_NSE_logNSE"
output_dir.mkdir(parents=True, exist_ok=True)
nc_files = list(data_path.glob("*.nc"))

n_samples = 12000
n_replicates = 10
n_jobs = get_safe_njobs(min_ram_per_job_gb=2.5)

# --- Run calibration in parallel ---
all_results = Parallel(n_jobs=n_jobs, verbose=5)(
    delayed(calibrate_catchment_replicate)(
        nc_file, n_samples,
        datetime.datetime(1999,1,1),
        datetime.datetime(2008,12,31),
        datetime.datetime(2009,1,1),
        datetime.datetime(2014,12,31),
        replicate_id,
        output_dir
    )
    for nc_file in nc_files
    for replicate_id in range(1, n_replicates+1)
)

# --- Save overall summary ---
summary_df = pd.DataFrame(all_results)

# Save per-catchment replicate summaries
for catchment, group in summary_df.groupby("catchment"):
    catchment_outdir = output_dir / catchment
    catchment_outdir.mkdir(parents=True, exist_ok=True)
    # Highlight replicate with highest logNSE
    best_idx = group["logNSE_val"].idxmax()
    group["best_logNSE"] = False
    group.loc[best_idx, "best_logNSE"] = True
    catchment_summary_path = catchment_outdir / f"{catchment}_replicates_summary.csv"
    group.to_csv(catchment_summary_path, index=False)
    print(f"✅ Saved {catchment} replicate summary to: {catchment_summary_path}")

# Save combined global summary
summary_path = output_dir / "best_parameters_GR4J_replicates.csv"
summary_df.to_csv(summary_path, index=False)
print(f"\n📄 Global summary saved to: {summary_path}")


🧠 16 physical cores / 24 logical cores detected.
💾 Total RAM: 63.7 GB | Available: 47.0 GB
⚙️  Using 16 parallel jobs (~2.5 GB/job)


[Parallel(n_jobs=16)]: Using backend LokyBackend with 16 concurrent workers.
[Parallel(n_jobs=16)]: Done  40 tasks      | elapsed:  1.1min
[Parallel(n_jobs=16)]: Done 130 tasks      | elapsed:  3.3min
[Parallel(n_jobs=16)]: Done 256 tasks      | elapsed:  5.9min
[Parallel(n_jobs=16)]: Done 418 tasks      | elapsed:  9.8min
[Parallel(n_jobs=16)]: Done 616 tasks      | elapsed: 14.3min
[Parallel(n_jobs=16)]: Done 850 tasks      | elapsed: 19.7min
[Parallel(n_jobs=16)]: Done 1120 tasks      | elapsed: 25.8min
[Parallel(n_jobs=16)]: Done 1426 tasks      | elapsed: 32.6min
[Parallel(n_jobs=16)]: Done 1768 tasks      | elapsed: 40.4min
[Parallel(n_jobs=16)]: Done 2146 tasks      | elapsed: 48.8min
[Parallel(n_jobs=16)]: Done 2560 tasks      | elapsed: 58.0min
[Parallel(n_jobs=16)]: Done 3010 tasks      | elapsed: 68.1min
[Parallel(n_jobs=16)]: Done 3496 tasks      | elapsed: 78.9min
[Parallel(n_jobs=16)]: Done 4018 tasks      | elapsed: 90.6min
[Parallel(n_jobs=16)]: Done 4576 tasks      | e

✅ Saved 10002 replicate summary to: C:\Users\jougahi001\OneDrive - University of Dundee\Hydrogr\hydrogr-master\hydrogr-master\example\GR4J_Calib_NSE_logNSE\10002\10002_replicates_summary.csv
✅ Saved 10003 replicate summary to: C:\Users\jougahi001\OneDrive - University of Dundee\Hydrogr\hydrogr-master\hydrogr-master\example\GR4J_Calib_NSE_logNSE\10003\10003_replicates_summary.csv
✅ Saved 1001 replicate summary to: C:\Users\jougahi001\OneDrive - University of Dundee\Hydrogr\hydrogr-master\hydrogr-master\example\GR4J_Calib_NSE_logNSE\1001\1001_replicates_summary.csv
✅ Saved 101002 replicate summary to: C:\Users\jougahi001\OneDrive - University of Dundee\Hydrogr\hydrogr-master\hydrogr-master\example\GR4J_Calib_NSE_logNSE\101002\101002_replicates_summary.csv
✅ Saved 101005 replicate summary to: C:\Users\jougahi001\OneDrive - University of Dundee\Hydrogr\hydrogr-master\hydrogr-master\example\GR4J_Calib_NSE_logNSE\101005\101005_replicates_summary.csv
✅ Saved 102001 replicate summary to: C:\Us